# RAG com IA e PDFs — versão com retrieval de verdade (chunking + embeddings)

Curso: IA Aplicada ao Desenvolvimento de Software para Servidores Públicos · Aula 8 (RAG)

Este é o companheiro do notebook `rag_enap_colab.ipynb` (que extrai o PDF inteiro e cola no prompt, sem retrieval). Aqui implementamos o fluxo completo ensinado nas Etapas 1 e 2 da aula:

```
PDF → extração → chunking → embeddings → índice → pergunta → embedding da pergunta →
similaridade → top-k → contexto recuperado → LLM → resposta + fontes
```

**O que muda em relação ao outro notebook:** em vez de mandar os ~69.000 tokens dos dois relatórios a cada pergunta, buscamos só os pedaços (chunks) mais relevantes — normalmente ~1.000 tokens — e mandamos só eles. Chunking, embeddings, índice e busca rodam **de graça**, localmente no Colab (biblioteca `sentence-transformers`); só a geração final da resposta (Passo 7) chama a API paga da OpenAI.

**Sobre os documentos:** os mesmos dois Relatórios de Gestão da Enap (2023 e 2024) do outro notebook — reais, públicos, documentos de prestação de contas.

## Passo 0 — instalar as bibliotecas

- `pypdf` — lê o texto de dentro de um PDF (sucessor do `PyPDF2`, mesma API básica).
- `sentence-transformers` — gera embeddings localmente, de graça, sem chamar nenhuma API paga.
- `numpy` — cálculo de similaridade de cosseno.
- `openai` — só é usada no Passo 7, para gerar a resposta final a partir dos trechos recuperados.
- `python-dotenv` — carrega a chave de API de um arquivo `.env`, em vez de deixá-la escrita no código.

In [ ]:
!pip install -q pypdf sentence-transformers numpy openai python-dotenv

## Passo 1 — enviar os dois relatórios

**Antes de rodar a célula abaixo**, baixe os dois PDFs no seu computador — pelos botões "Baixar Relatório de Gestão 2023/2024" na própria aula, ou direto destes links: [Relatório 2023](https://repositorio.enap.gov.br/handle/1/9959) e [Relatório 2024](https://repositorio.enap.gov.br/handle/1/8854).

**Por que upload, e não a célula buscar sozinha**: o servidor do repositório da Enap bloqueia pedidos automáticos vindos de faixas de IP de datacenter do Google (`HTTPError: 403 Forbidden`, mesmo o link funcionando normalmente em qualquer navegador). Pedir upload evita esse problema por completo.

Rode a célula abaixo e selecione os dois PDFs baixados de uma vez (Ctrl/Cmd + clique para marcar os dois).

In [ ]:
from google.colab import files

print("Envie os dois arquivos baixados: o Relatório de Gestão 2023 e o Relatório de Gestão 2024 da Enap.")
enviados = files.upload()

pdf_bytes_por_ano = {}
for nome_arquivo, conteudo in enviados.items():
    if "2023" in nome_arquivo:
        pdf_bytes_por_ano["2023"] = conteudo
    elif "2024" in nome_arquivo:
        pdf_bytes_por_ano["2024"] = conteudo
    else:
        print(f"Aviso: não reconheci '{nome_arquivo}' como 2023 ou 2024 pelo nome do arquivo.")

faltando = {"2023", "2024"} - set(pdf_bytes_por_ano.keys())
if faltando:
    raise RuntimeError(
        f"Envie os relatórios de {' e '.join(sorted(faltando))} para continuar "
        "(mantenha '2023'/'2024' em algum lugar do nome do arquivo)."
    )

print("Recebidos:", list(pdf_bytes_por_ano.keys()))

## Passo 2 — extrair o texto

Mesma ideia do outro notebook, com uma diferença importante: nem toda página de PDF tem texto extraível (páginas só com imagem/escaneadas, por exemplo). Sem tratar isso, `pagina.extract_text()` pode devolver `None`, e `texto += None` quebra com `TypeError`. A linha `texto_pagina = pagina.extract_text() or ""` evita isso — testado nos dois relatórios reais desta aula: o de 2024 tem uma página sem texto extraível, e a extração não quebra.

In [ ]:
import io
from pypdf import PdfReader

def extrair_texto_pdf(conteudo_pdf):
    leitor = PdfReader(io.BytesIO(conteudo_pdf))
    texto = ""
    paginas_sem_texto = 0
    for pagina in leitor.pages:
        texto_pagina = pagina.extract_text() or ""  # nunca soma None a uma string
        if not texto_pagina.strip():
            paginas_sem_texto += 1
        texto += texto_pagina + "\n"
    return texto, paginas_sem_texto

texto_por_ano = {}
for ano, conteudo in pdf_bytes_por_ano.items():
    texto, vazias = extrair_texto_pdf(conteudo)
    texto_por_ano[ano] = texto
    print(f"Relatório {ano}: {len(texto)} caracteres extraídos, {vazias} página(s) sem texto extraível")

## Passo 3 — chunking: dividir os documentos em pedaços

**Tamanho do chunk:** pedaço pequeno demais (poucas dezenas de caracteres) perde contexto — uma frase cortada no meio não diz mais do que está falando. Pedaço grande demais dilui o que é relevante — o embedding vira uma média de vários assuntos misturados, e a busca fica menos precisa.

**Overlap (sobreposição):** sem sobreposição, uma frase que atravessa a fronteira entre dois chunks fica partida nos dois, sem sentido completo em nenhum. Repetir um pedaço do fim de um chunk no começo do próximo reduz esse problema.

Usamos aqui 800 caracteres por chunk com 100 de sobreposição — um ponto de partida razoável para texto em português, não um valor universalmente correto (a seção seguinte volta a esse ponto).

In [ ]:
def dividir_em_chunks(texto, tamanho=800, sobreposicao=100):
    chunks = []
    inicio = 0
    n = len(texto)
    while inicio < n:
        fim = min(inicio + tamanho, n)
        chunk = texto[inicio:fim].strip()
        if chunk:
            chunks.append(chunk)
        if fim == n:
            break
        inicio = fim - sobreposicao  # sobreposição: o próximo chunk repete o fim deste
    return chunks

todos_chunks = []  # cada item: {"id", "ano", "texto"}
for ano, texto in texto_por_ano.items():
    for i, texto_chunk in enumerate(dividir_em_chunks(texto, tamanho=800, sobreposicao=100)):
        todos_chunks.append({"id": f"{ano}-{i}", "ano": ano, "texto": texto_chunk})
    print(f"Relatório {ano}: {sum(1 for c in todos_chunks if c['ano'] == ano)} chunks")

print(f"Total de chunks nos dois documentos: {len(todos_chunks)}")

## Passo 4 — embeddings: transformar cada chunk num vetor

Um **embedding** é uma lista de números (um vetor) que representa o *significado* de um texto — textos com sentido parecido geram vetores próximos entre si, mesmo usando palavras diferentes. Usamos o modelo `paraphrase-multilingual-MiniLM-L12-v2`, da biblioteca `sentence-transformers`: pequeno o suficiente para rodar no Colab gratuito, e treinado para várias línguas, incluindo português.

Isso roda **localmente, de graça** — diferente da chamada à OpenAI do Passo 7, não consome nenhum crédito pago. É o mesmo tipo de embedding local que a versão anterior deste material já usava (veja o README desta pasta).

In [ ]:
from sentence_transformers import SentenceTransformer

modelo_embeddings = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

textos_chunks = [c["texto"] for c in todos_chunks]
vetores = modelo_embeddings.encode(textos_chunks, show_progress_bar=True, batch_size=32)

for chunk, vetor in zip(todos_chunks, vetores):
    chunk["embedding"] = vetor

print(f"{len(vetores)} embeddings gerados, dimensão {vetores.shape[1]} cada.")

## Passo 5 — o índice, e a similaridade de cosseno

Aqui o "índice vetorial" é só a lista `todos_chunks`, cada item já carregando seu embedding — para um acervo deste tamanho (algumas centenas de chunks), isso é suficiente; um vector database de verdade (FAISS, Pinecone, pgvector) só passa a valer a pena com volumes muito maiores.

A **similaridade de cosseno** mede o quão parecidos dois vetores são, olhando só para a direção deles (não o tamanho): varia de -1 (opostos) a 1 (idênticos). É a métrica mais comum para comparar embeddings de texto.

In [ ]:
import numpy as np

def similaridade_cosseno(a, b):
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

## Passo 6 — retrieval: da pergunta aos top-k chunks

`TOP_K` controla quantos chunks são recuperados por pergunta — não existe um valor universalmente correto: poucos chunks arriscam deixar de fora um trecho relevante; muitos chunks aumentam o custo e o risco de "lost in the middle" (Etapa 5 da aula). Cinco é um ponto de partida razoável para este acervo.

Repare que os scores abaixo não são um "sim/não" limpo — são uma ordenação por proximidade. Uma pergunta genuinamente fora do escopo dos documentos (testada no Passo 8) ainda recebe algum score, só que mais baixo e menos consistente entre os top-k; o valor exato depende do modelo de embedding, do corpus e de como os chunks foram cortados — não é um limiar fixo e confiável sozinho.

In [ ]:
TOP_K = 5

def buscar(pergunta, top_k=TOP_K):
    embedding_pergunta = modelo_embeddings.encode([pergunta])[0]
    resultados = [
        (chunk, similaridade_cosseno(embedding_pergunta, chunk["embedding"]))
        for chunk in todos_chunks
    ]
    resultados.sort(key=lambda par: par[1], reverse=True)
    return resultados[:top_k]

def mostrar_resultados(resultados):
    for chunk, score in resultados:
        preview = chunk["texto"][:110].replace("\n", " ")
        print(f"[Relatório {chunk['ano']}, chunk {chunk['id']}] score={score:.3f}")
        print(f"  {preview}...")

pergunta_teste = "Quantos participantes o Congresso do CLAD reuniu, e em qual ano isso aconteceu?"
resultados = buscar(pergunta_teste)
print(f"Pergunta: {pergunta_teste}\n")
mostrar_resultados(resultados)

## Passo 7 — configurar a chave de API e gerar a resposta

Só agora — depois de já ter encontrado os trechos certos, de graça — entra a chamada paga à OpenAI. O prompt final usa **só os chunks recuperados no Passo 6**, não os documentos inteiros: essa é a diferença central em relação ao notebook `rag_enap_colab.ipynb`.

In [ ]:
import os
from getpass import getpass
from dotenv import load_dotenv

chave_api = getpass("Cole sua chave da API OpenAI: ")
with open(".env", "w") as f:
    f.write(f"OPENAI_API_KEY={chave_api}\n")
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("A chave da API OpenAI não foi encontrada. Rode a célula de novo.")
print("Chave carregada — pronta para uso.")

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

def perguntar_com_retrieval(pergunta, top_k=TOP_K, mostrar_chunks=True):
    resultados = buscar(pergunta, top_k=top_k)
    if mostrar_chunks:
        print(f"Chunks recuperados para: {pergunta}\n")
        mostrar_resultados(resultados)
        print()

    contexto_recuperado = "\n\n".join(
        f"[Fonte: Relatório {c['ano']}, trecho {c['id']}, score {s:.2f}]\n{c['texto']}"
        for c, s in resultados
    )
    prompt = (
        "Responda à pergunta usando SOMENTE os trechos recuperados abaixo. "
        "Se a resposta não estiver nos trechos, diga explicitamente que não encontrou "
        "informação suficiente nas fontes recuperadas — não invente.\n\n"
        f"Trechos recuperados:\n{contexto_recuperado}\n\n"
        f"Pergunta: {pergunta}\nResposta:"
    )
    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=250,
    )
    resposta = completion.choices[0].message.content
    fontes = sorted({f"Relatório {c['ano']}" for c, s in resultados})
    print(f"Resposta: {resposta}\n")
    print(f"Fontes citadas na busca: {', '.join(fontes)}")
    return resposta

_ = perguntar_com_retrieval("Quantos participantes o Congresso do CLAD reuniu, e em qual ano isso aconteceu?")

## Passo 8 — teste de relevância: uma pergunta claramente fora do escopo

RAG reduz o risco de invenção, mas não é uma garantia — o teste que prova isso é perguntar algo que certamente **não** está nos relatórios. Se o retrieval funcionar bem, os scores devem ser visivelmente mais fracos/inconsistentes do que os da pergunta anterior, e a resposta deve admitir que não encontrou a informação, em vez de inventar um número.

In [ ]:
_ = perguntar_com_retrieval("Qual é a receita de venda de picolés da Enap em 2024?")

## Comparação de custo/contexto (números reais deste notebook)

Medido nos dois relatórios reais desta aula, com chunk de 800 caracteres:

| | Documento inteiro | RAG (top-5 chunks) |
|---|---|---|
| Chunks/documentos enviados | os 2 relatórios completos | 5 de 395 chunks |
| Caracteres enviados por pergunta | ~276.000 | ~4.000 |
| Tokens aproximados por pergunta | ~69.000 | ~1.000 |

Esses números são deste acervo específico (chunk de 800 caracteres, `TOP_K=5`) — mudam com outro tamanho de chunk, outro `TOP_K`, ou outro corpus. O ponto pedagógico não é o número exato, é a ordem de grandeza: retrieval manda uma fração pequena do que existe, não tudo.

## O que este notebook mostra (e a diferença para `rag_enap_colab.ipynb`)

1. **Chunking** ✅ — Passo 3, os dois relatórios viram 395 pedaços de ~800 caracteres.
2. **Embeddings** ✅ — Passo 4, cada chunk vira um vetor de 384 números, localmente, de graça.
3. **Índice** ✅ — Passo 5, a lista `todos_chunks` com embeddings anexados.
4. **Embedding da pergunta** ✅ — dentro de `buscar()`, Passo 6.
5. **Similaridade** ✅ — `similaridade_cosseno()`, Passo 5.
6. **Top-k** ✅ — `TOP_K = 5`, ajustável, Passo 6.
7. **Retrieval** ✅ — `buscar()` ordena por score e recorta os `top_k`, Passo 6.
8. **Contexto recuperado, não o documento inteiro** ✅ — o prompt do Passo 7 usa só os chunks retornados por `buscar()`.
9. **LLM** ✅ — chamada à OpenAI no Passo 7, só depois de já ter os trechos certos.
10. **Fontes** ✅ — cada resposta imprime de qual(is) relatório(s) os chunks usados vieram.

**Limite importante, sem exagero:** mesmo com retrieval funcionando bem, a resposta final ainda passa por um LLM, que pode interpretar mal um trecho corretamente recuperado — RAG reduz o risco de a IA inventar uma fonte que não existe, mas não garante que a interpretação do trecho encontrado esteja certa. A resposta ainda precisa ser conferida por uma pessoa, principalmente antes de qualquer uso oficial.

**Comparando com o outro notebook** (`rag_enap_colab.ipynb`, documento inteiro no prompt): aquele é mais simples de programar e entender, e funciona bem quando o documento cabe na janela de contexto do modelo — mas manda o texto inteiro a cada pergunta, custando mais tokens e não escalando para um acervo muito maior. Este aqui é mais trabalhoso de montar, mas é o padrão que de fato se chama RAG com retrieval — e é o que a maioria das ferramentas de produção (incluindo o NotebookLM usado no Mão na massa desta aula) faz por baixo dos panos.